# NLP Model: Multi-Label Call Code Classification

Trains two models to classify `call_transcript` into `call_codes` (32 labels, multi-label).  

| Model | Type | Use case |
|---|---|---|
| Model 1 | CountVectorizer + Logistic Regression | Baseline, lightweight, easy Spark deploy |
| Model 2 | RoBERTa fine-tuned | Higher precision, used as primary model |

**Output:** `models/bow_model.pkl`, `models/label_classes.json`, `models/roberta_saved/`  
All artifacts are ready to load into the PySpark Silver layer ETL job.

## 1. Setup & Paths

In [ ]:
import os
import json
import warnings
import joblib
import numpy as np
import pandas as pd
from pathlib import Path

warnings.filterwarnings('ignore')

# Paths relative to this notebook — works on any OS
DATA_DIR  = Path('.')         # train.csv / valid.csv / test.csv
MODEL_DIR = Path('models')    # saved model artifacts
MODEL_DIR.mkdir(exist_ok=True)

print(f'Data  dir : {DATA_DIR.resolve()}')
print(f'Model dir : {MODEL_DIR.resolve()}')

## 2. Load Data & Label Encoding

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer

print('Loading data...')
train_df = pd.read_csv(DATA_DIR / 'train.csv')
valid_df = pd.read_csv(DATA_DIR / 'valid.csv')
test_df  = pd.read_csv(DATA_DIR / 'test.csv')
print(f'Train: {len(train_df):,}  |  Valid: {len(valid_df):,}  |  Test: {len(test_df):,}')

def process_labels(text):
    if pd.isna(text):
        return []
    return [l.strip() for l in text.split(',')]

for df in [train_df, valid_df, test_df]:
    df['call_code_list'] = df['call_code'].apply(process_labels)

# BoW uses merged train+valid for a larger vocabulary
# RoBERTa uses separate splits for proper validation
full_train_df = pd.concat([train_df, valid_df], ignore_index=True)

mlb = MultiLabelBinarizer()
mlb.fit(full_train_df['call_code_list'])

y_train_full = mlb.transform(full_train_df['call_code_list'])
y_train      = mlb.transform(train_df['call_code_list']).astype('float32')
y_valid      = mlb.transform(valid_df['call_code_list']).astype('float32')
y_test       = mlb.transform(test_df['call_code_list'])

print(f'Labels ({len(mlb.classes_)}): {list(mlb.classes_)}')

---
## Model 1 — Bag of Words + Logistic Regression

Baseline model. Fast to train, runs in-process on every Spark executor via broadcast.  
Uses `CountVectorizer` (unigrams + bigrams, 5k features) + `OneVsRestClassifier(LogisticRegression)`.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, f1_score,
    precision_score, recall_score, accuracy_score,
)

print('Building vocabulary...')
counter = CountVectorizer(max_features=5000, ngram_range=(1, 2), stop_words='english')
X_train_count = counter.fit_transform(full_train_df['call_transcript'])
X_test_count  = counter.transform(test_df['call_transcript'])
print(f'Vocabulary size: {X_train_count.shape[1]:,}')

print('Training Logistic Regression (OneVsRest)...')
base_lr        = LogisticRegression(solver='liblinear', class_weight='balanced', random_state=42)
counting_model = OneVsRestClassifier(base_lr)
counting_model.fit(X_train_count, y_train_full)

y_pred_bow = counting_model.predict(X_test_count)

f1_mi = f1_score(y_test, y_pred_bow, average='micro')
f1_ma = f1_score(y_test, y_pred_bow, average='macro')
prec  = precision_score(y_test, y_pred_bow, average='micro')
rec   = recall_score(y_test, y_pred_bow, average='micro')
em    = accuracy_score(y_test, y_pred_bow)

sep = '=' * 55
print(f'\n{sep}')
print('  MODEL 1 — CountVectorizer + Logistic Regression')
print(sep)
print(f'  F1 Micro    : {f1_mi*100:.2f}%')
print(f'  F1 Macro    : {f1_ma*100:.2f}%')
print(f'  Precision   : {prec*100:.2f}%')
print(f'  Recall      : {rec*100:.2f}%')
print(f'  Exact Match : {em*100:.2f}%')
print(sep)
print(classification_report(y_test, y_pred_bow, target_names=mlb.classes_, zero_division=0))

In [ ]:
# Save BoW model as a single bundle: vectorizer + classifier + label binarizer
bow_bundle = {
    'vectorizer': counter,
    'classifier': counting_model,
    'mlb': mlb,
}
bow_path = MODEL_DIR / 'bow_model.pkl'
joblib.dump(bow_bundle, bow_path)
print(f'BoW model saved      -> {bow_path}')

# Save label list as JSON (human-readable, no pickle needed)
labels_path = MODEL_DIR / 'label_classes.json'
with open(labels_path, 'w', encoding='utf-8') as f:
    json.dump(mlb.classes_.tolist(), f, indent=2, ensure_ascii=False)
print(f'Label classes saved  -> {labels_path}')

---
## Model 2 — RoBERTa Fine-tuned

Transformer model using `roberta-base`.  
Higher precision than BoW (80.78% vs 65.92%) — fewer false positives on call code assignment.  
Uses `BCEWithLogitsLoss` for multi-label output, custom `MyTrainer` to handle `float32` labels.  
> Requires GPU for reasonable training time (`batch_size=4`, `3 epochs`).

In [ ]:
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch version : {torch.__version__}')
print(f'Device          : {device}')

MODEL_NAME = 'roberta-base'
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)

# Assign label arrays to dataframe columns before building Dataset
train_df['labels'] = list(y_train)
valid_df['labels'] = list(y_valid)
test_df['labels']  = list(y_test.astype('float32'))

train_dataset = Dataset.from_pandas(train_df[['call_transcript', 'labels']])
valid_dataset = Dataset.from_pandas(valid_df[['call_transcript', 'labels']])
test_dataset  = Dataset.from_pandas(test_df[['call_transcript', 'labels']])

def tokenize(batch):
    return tokenizer(
        batch['call_transcript'],
        truncation=True,
        padding='max_length',
        max_length=512,
    )

train_dataset = train_dataset.map(tokenize, batched=True)
valid_dataset = valid_dataset.map(tokenize, batched=True)
test_dataset  = test_dataset.map(tokenize, batched=True)

train_dataset.set_format(type='torch')
valid_dataset.set_format(type='torch')
test_dataset.set_format(type='torch')

print(f'Dataset — Train: {len(train_dataset):,}  Valid: {len(valid_dataset):,}  Test: {len(test_dataset):,}')

In [ ]:
num_labels = len(mlb.classes_)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    problem_type='multi_label_classification',
)
model.to(device)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs  = torch.sigmoid(torch.tensor(logits)).numpy()
    preds  = (probs >= 0.5).astype(int)
    labels = labels.astype(int)
    return {
        'f1_micro'  : f1_score(labels, preds, average='micro'),
        'f1_macro'  : f1_score(labels, preds, average='macro'),
        'precision' : precision_score(labels, preds, average='micro'),
        'recall'    : recall_score(labels, preds, average='micro'),
        'accuracy'  : accuracy_score(labels, preds),
    }

training_args = TrainingArguments(
    output_dir                  = str(MODEL_DIR / 'roberta_checkpoints'),
    learning_rate               = 2e-5,
    per_device_train_batch_size = 4,
    per_device_eval_batch_size  = 4,
    num_train_epochs            = 3,
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',
    logging_steps               = 50,
    load_best_model_at_end      = True,
)

class MyTrainer(Trainer):
    """Custom trainer that casts labels to float32 for BCEWithLogitsLoss."""
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.get('labels').float()
        outputs = model(**inputs)
        loss    = torch.nn.BCEWithLogitsLoss()(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

trainer = MyTrainer(
    model            = model,
    args             = training_args,
    train_dataset    = train_dataset,
    eval_dataset     = valid_dataset,
    processing_class = tokenizer,
    compute_metrics  = compute_metrics,
)

trainer.train()

In [ ]:
# Evaluate using batch prediction (faster than per-sample loop)
print('Predicting on test set...')
pred_output = trainer.predict(test_dataset)
probs  = torch.sigmoid(torch.tensor(pred_output.predictions)).numpy()
y_pred = (probs >= 0.5).astype(int)
y_true = pred_output.label_ids.astype(int)

f1_mi = f1_score(y_true, y_pred, average='micro')
f1_ma = f1_score(y_true, y_pred, average='macro')
prec  = precision_score(y_true, y_pred, average='micro')
rec   = recall_score(y_true, y_pred, average='micro')
em    = accuracy_score(y_true, y_pred)

sep = '=' * 55
print(f'\n{sep}')
print('  MODEL 2 — RoBERTa fine-tuned')
print(sep)
print(f'  F1 Micro    : {f1_mi*100:.2f}%')
print(f'  F1 Macro    : {f1_ma*100:.2f}%')
print(f'  Precision   : {prec*100:.2f}%')
print(f'  Recall      : {rec*100:.2f}%')
print(f'  Exact Match : {em*100:.2f}%')
print(sep)

# Save final model + tokenizer
roberta_path = MODEL_DIR / 'roberta_saved'
model.save_pretrained(roberta_path)
tokenizer.save_pretrained(roberta_path)
print(f'\nRoBERTa model saved  -> {roberta_path}')
print(f'Tokenizer saved      -> {roberta_path}')

---
## PySpark Integration — Silver Layer ETL

The BoW model (`bow_model.pkl`) is the recommended choice for Spark:  
it is lightweight, serializable, and runs on every executor via **broadcast + Pandas UDF**.

Copy the snippet below into `project/batch-etl/silver_nlp.py`.

In [ ]:
# =============================================================================
# FILE: project/batch-etl/silver_nlp.py
# PURPOSE: Apply trained BoW NLP model inside PySpark Silver ETL
# =============================================================================
#
# from pyspark.sql import SparkSession
# from pyspark.sql import functions as F
# from pyspark.sql.types import ArrayType, StringType
# import pandas as pd
# import joblib
# import os
#
# # Path inside the Spark container (volume-mounted from ./batch-etl)
# MODELS_PATH = '/opt/spark/work-dir/batch-etl/models'
#
# spark = SparkSession.builder.appName('silver_nlp').getOrCreate()
#
# # 1. Load model once and broadcast to all executors
# bow_bundle = spark.sparkContext.broadcast(
#     joblib.load(os.path.join(MODELS_PATH, 'bow_model.pkl'))
# )
#
# # 2. Pandas UDF — Spark calls this on batches of rows, not row-by-row
# @F.pandas_udf(ArrayType(StringType()))
# def predict_call_codes(transcripts: pd.Series) -> pd.Series:
#     bundle = bow_bundle.value
#     X      = bundle['vectorizer'].transform(transcripts.fillna(''))
#     preds  = bundle['classifier'].predict(X)
#     labels = bundle['mlb'].inverse_transform(preds)
#     return pd.Series([list(l) for l in labels])
#
# # 3. Apply on Silver DataFrame
# df_silver = df_silver.withColumn(
#     'call_codes_predicted',
#     predict_call_codes(F.col('call_transcript'))
# )

# ── Summary of saved artifacts ───────────────────────────────────────────────
print('Model artifacts saved to:')
print(f'  {MODEL_DIR / "bow_model.pkl"}        <- broadcast in Spark Pandas UDF')
print(f'  {MODEL_DIR / "label_classes.json"}   <- human-readable label list')
print(f'  {MODEL_DIR / "roberta_saved/"}       <- load with from_pretrained()')
print(f'  {MODEL_DIR / "roberta_checkpoints/"} <- training checkpoints (can delete after saving)')